# CMR Multi-Parameter Sweep

**Behavioral readouts** (produced for every sweep):
recall accuracy, SPC, PFR, lag-CRP (split ℓ<0 / ℓ>0),
conditional forward & backward lag rates,
unconditional |ℓ| summaries.

**Parameter-specific internal diagnostics** per sweep.

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings('ignore')

from cmr.config import (
    N, BASE_PARAMS,
    B_rec_grid, gamma_fc_grid, eta_grid, B_encD_scale_grid,
    n_sims,
)
from cmr.sweep import sweep_one_param
from cmr.utils import _get_sweep_entry

# Metrics
from cmr.metrics import (
    recall_accuracy, compute_spc, compute_pfr,
    compute_lag_crp, lag_crp_with_counts,
    conditional_forward_lag_rates, conditional_backward_lag_rates,
    unconditional_transition_summaries,
)

# Recall-stage diagnostics
from cmr.diagnostics_recall import (
    get_trace_sims, mean_curve_over_sims, mean_vector_over_sims,
    item_evidence_asymmetry_means, fc_alignment_asymmetry_means,
)

# Encoding-stage diagnostics
from cmr.diagnostics_encoding import (
    sweep_matrix_norms, sweep_w_fc_lag_strength_profile,
    sweep_w_fc_forward_backward_lag_asymmetry, sweep_recall_counts,
)

# Visualization — shared helpers
from cmr.visualization import (
    make_sweep_colors, _add_colorbar,
    line_with_colored_points, plot_split_lags,
)

# Visualization — individual behavioral plots
from cmr.visualization import (
    plot_recall_accuracy, plot_spc_sweep, plot_pfr_heatmap,
    plot_lag_crp_sweep, plot_directional_lag_rates,
    plot_unconditional_lag_summaries,
)

# Visualization — recall-stage diagnostics
from cmr.visualization import (
    plot_lag_crp_diagnostics,
    plot_item_evidence_asymmetry_paired,
    plot_fc_alignment_asymmetry_sweep,
    plot_fc_alignment_asymmetry_curves,
    plot_cosine_similarity_sweep,
    plot_mean_evidence_by_pos,
    plot_scalar_metric_vs_param,
)

# Visualization — encoding-stage diagnostics
from cmr.visualization import (
    plot_w_fc_lag_strength_profile_sweep,
    plot_matrix_norms_sweep,
    plot_w_fc_forward_backward_lag_asymmetry_sweep,
)

### Configuration

All defaults live in `cmr/config.py`.  Override below before running.

In [ ]:
# Uncomment to override:
# BASE_PARAMS["B_rec"] = 0.55
# n_sims = 500

---
## 1. Sweep retrieval context drift $B_{rec}$

In [ ]:
sweep_B_rec = sweep_one_param(
    param_name="B_rec",
    param_grid=B_rec_grid,
    base_params=BASE_PARAMS,
    n_sims=n_sims,
    collect_diagnostics=True,
    recency_k=3,
)

### Behavioral metrics: accuracy

In [ ]:
plot_recall_accuracy(sweep_B_rec, B_rec_grid, r"$B_{rec}$")

### Behavioral metrics: SPC

In [ ]:
plot_spc_sweep(sweep_B_rec, B_rec_grid, r"$B_{rec}$")

### Behavioral metrics: PFR

In [ ]:
plot_pfr_heatmap(sweep_B_rec, B_rec_grid, r"$B_{rec}$")

### Behavioral metrics: lag-CRP

In [ ]:
plot_lag_crp_sweep(sweep_B_rec, B_rec_grid, r"$B_{rec}$")

### Behavioral metrics: conditional lag rates

In [ ]:
plot_directional_lag_rates(sweep_B_rec, B_rec_grid, r"$B_{rec}$")

### Behavioral metrics: unconditional lag summaries

In [ ]:
plot_unconditional_lag_summaries(sweep_B_rec, B_rec_grid, r"$B_{rec}$")

### Model Internal Diagnostics ($B_{rec}$)

Retrieval context update strength — most informative internals are
context-update sanity (radicand, norm, cos), evidence profiles,
and neighbor asymmetries.

#### Radicand $\mathrm{rad}(t)$ across retrieval steps

In [ ]:
param_grid = np.asarray(B_rec_grid, dtype=float)
colors, norm, cmap = make_sweep_colors(param_grid)

fig, ax = plt.subplots(figsize=(8, 5))
for val, c in zip(param_grid, colors):
    ts = get_trace_sims(sweep_B_rec, val)
    x, rad = mean_curve_over_sims(ts, "rad")
    if x.size:
        ax.plot(x, rad, marker="o", color=c, ms=4)
ax.axhline(0, ls="--", alpha=0.6)
ax.set_title(r"Radicand $\mathrm{rad}(t)$ across $B_{rec}$")
ax.set_xlabel("recall step"); ax.set_ylabel(r"$\mathrm{rad}$")
ax.grid(alpha=0.3)
_add_colorbar(fig, ax, norm, cmap, r"$B_{rec}$")
fig.tight_layout(); plt.show()

#### Context norm $\|c\|$ after each retrieval update

In [ ]:
param_grid = np.asarray(B_rec_grid, dtype=float)
colors, norm, cmap = make_sweep_colors(param_grid)

fig, ax = plt.subplots(figsize=(8, 5))
for val, c in zip(param_grid, colors):
    ts = get_trace_sims(sweep_B_rec, val)
    x, cn = mean_curve_over_sims(ts, "c_norm")
    if x.size:
        ax.plot(x, cn, marker="o", color=c, ms=4)
ax.axhline(1, ls="--", alpha=0.6)
ax.set_title(r"Context norm $\|c\|$ after update across $B_{rec}$")
ax.set_xlabel("recall step"); ax.set_ylabel(r"$\|c\|$")
ax.grid(alpha=0.3)
_add_colorbar(fig, ax, norm, cmap, r"$B_{rec}$")
fig.tight_layout(); plt.show()

#### Cosine similarity $\cos(c, c_{in})$ after update

In [ ]:
plot_cosine_similarity_sweep(sweep_B_rec, B_rec_grid, param_name=r"$B_{rec}$")

#### Mean evidence $f_{in}(i)$ by serial position

In [ ]:
plot_mean_evidence_by_pos(sweep_B_rec, B_rec_grid, param_name=r"$B_{rec}$")

#### Neighbor item-evidence asymmetry during selection

In [ ]:
plot_item_evidence_asymmetry_paired(sweep_B_rec, B_rec_grid, param_name=r"$B_{rec}$")

#### Neighbor context-to-item-input alignment ($\Delta_{FC}$) asymmetry during updating

In [ ]:
plot_fc_alignment_asymmetry_sweep(sweep_B_rec, B_rec_grid, param_name=r"$B_{rec}$")

#### lag-CRP diagnostics (numerator & denominator)

In [ ]:
plot_lag_crp_diagnostics(sweep_B_rec, B_rec_grid, param_name=r"$B_{rec}$")

---
## 2. Sweep feature-to-context association strength $\gamma_{fc}$

In [ ]:
sweep_gamma_fc = sweep_one_param(
    param_name="gamma_fc",
    param_grid=gamma_fc_grid,
    base_params=BASE_PARAMS,
    n_sims=n_sims,
    collect_diagnostics=True,
    recency_k=3,
)

### Behavioral metrics: accuracy

In [ ]:
plot_recall_accuracy(sweep_gamma_fc, gamma_fc_grid, r"$\gamma_{fc}$")

### Behavioral metrics: SPC

In [ ]:
plot_spc_sweep(sweep_gamma_fc, gamma_fc_grid, r"$\gamma_{fc}$")

### Behavioral metrics: PFR

In [ ]:
plot_pfr_heatmap(sweep_gamma_fc, gamma_fc_grid, r"$\gamma_{fc}$")

### Behavioral metrics: lag-CRP

In [ ]:
plot_lag_crp_sweep(sweep_gamma_fc, gamma_fc_grid, r"$\gamma_{fc}$")

### Behavioral metrics: conditional lag rates

In [ ]:
plot_directional_lag_rates(sweep_gamma_fc, gamma_fc_grid, r"$\gamma_{fc}$")

### Behavioral metrics: unconditional lag summaries

In [ ]:
plot_unconditional_lag_summaries(sweep_gamma_fc, gamma_fc_grid, r"$\gamma_{fc}$")

### Model Internal Diagnostics ($\gamma_{fc}$)

FC learning rate — most informative internals are learned $M_{FC}$ structure,
item-evidence asymmetry, lag-CRP numerator/denominator, and evidence profiles.

#### $M_{FC}$ lag-strength profile

In [ ]:
plot_w_fc_lag_strength_profile_sweep(sweep_gamma_fc, gamma_fc_grid, r"$\gamma_{fc}$", matrix_key="net_w_fc")

#### $M_{FC}$ norms

In [ ]:
plot_matrix_norms_sweep(sweep_gamma_fc, gamma_fc_grid, r"$\gamma_{fc}$", matrix_key="net_w_fc")

#### $M_{FC}$ forward/backward lag asymmetry

In [ ]:
plot_w_fc_forward_backward_lag_asymmetry_sweep(sweep_gamma_fc, gamma_fc_grid, r"$\gamma_{fc}$", matrix_key="net_w_fc")

#### Neighbor item-evidence asymmetry during selection

In [ ]:
plot_item_evidence_asymmetry_paired(sweep_gamma_fc, gamma_fc_grid, param_name=r"$\gamma_{fc}$")

#### Neighbor context-to-item-input alignment ($\Delta_{FC}$) asymmetry during updating

In [ ]:
plot_fc_alignment_asymmetry_sweep(sweep_gamma_fc, gamma_fc_grid, param_name=r"$\gamma_{fc}$")

#### $\Delta_{FC}$ side-by-side: $\gamma_{fc}$ and $B_{rec}$

In [ ]:
m_all_g, m_fwd_g, m_bwd_g = fc_alignment_asymmetry_means(sweep_gamma_fc, gamma_fc_grid)
m_all_b, m_fwd_b, m_bwd_b = fc_alignment_asymmetry_means(sweep_B_rec, B_rec_grid)

gc, gn, gm = make_sweep_colors(gamma_fc_grid)
bc, bn, bm = make_sweep_colors(B_rec_grid)

fig, axs = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)
plot_fc_alignment_asymmetry_curves(
    axs[0], gamma_fc_grid, m_all_g, m_fwd_g, m_bwd_g, gc,
    title=r"$\Delta_{FC}$ vs $\gamma_{fc}$", xlabel=r"$\gamma_{fc}$")
plot_fc_alignment_asymmetry_curves(
    axs[1], B_rec_grid, m_all_b, m_fwd_b, m_bwd_b, bc,
    title=r"$\Delta_{FC}$ vs $B_{rec}$", xlabel=r"$B_{rec}$")

sm0 = mpl.cm.ScalarMappable(norm=gn, cmap=gm); sm0.set_array([])
sm1 = mpl.cm.ScalarMappable(norm=bn, cmap=bm); sm1.set_array([])
fig.colorbar(sm0, ax=axs[0], pad=0.02).set_label(r"$\gamma_{fc}$")
fig.colorbar(sm1, ax=axs[1], pad=0.02).set_label(r"$B_{rec}$")
plt.show()

#### Mean $f_{in}(i)$ by serial position

In [ ]:
plot_mean_evidence_by_pos(sweep_gamma_fc, gamma_fc_grid, param_name=r"$\gamma_{fc}$")

#### lag-CRP diagnostics (numerator & denominator)

In [ ]:
plot_lag_crp_diagnostics(sweep_gamma_fc, gamma_fc_grid, param_name=r"$\gamma_{fc}$")

#### Cosine similarity $\cos(c, c_{in})$ after update

In [ ]:
plot_cosine_similarity_sweep(sweep_gamma_fc, gamma_fc_grid, param_name=r"$\gamma_{fc}$")

---
## 3. Sweep accumulator noise $\eta$

In [ ]:
sweep_eta = sweep_one_param(
    param_name="eta",
    param_grid=eta_grid,
    base_params=BASE_PARAMS,
    n_sims=n_sims,
    collect_diagnostics=True,
    recency_k=3,
)

### Behavioral metrics: accuracy

In [ ]:
plot_recall_accuracy(sweep_eta, eta_grid, r"$\eta$")

### Behavioral metrics: SPC

In [ ]:
plot_spc_sweep(sweep_eta, eta_grid, r"$\eta$")

### Behavioral metrics: PFR

In [ ]:
plot_pfr_heatmap(sweep_eta, eta_grid, r"$\eta$")

### Behavioral metrics: lag-CRP

In [ ]:
plot_lag_crp_sweep(sweep_eta, eta_grid, r"$\eta$")

### Behavioral metrics: conditional lag rates

In [ ]:
plot_directional_lag_rates(sweep_eta, eta_grid, r"$\eta$")

### Behavioral metrics: unconditional lag summaries

In [ ]:
plot_unconditional_lag_summaries(sweep_eta, eta_grid, r"$\eta$")

### Model Internal Diagnostics ($\eta$)

Accumulator noise — most informative internals are evidence entropy/recency mass
(which should be stable if $\eta$ only affects competition, not evidence),
evidence-by-position profiles, and asymmetries.

#### Cosine similarity $\cos(c, c_{in})$ after update

In [ ]:
plot_cosine_similarity_sweep(sweep_eta, eta_grid, param_name=r"$\eta$")

#### Mean $f_{in}(i)$ by serial position

In [ ]:
plot_mean_evidence_by_pos(sweep_eta, eta_grid, param_name=r"$\eta$")

#### Neighbor item-evidence asymmetry during selection

In [ ]:
plot_item_evidence_asymmetry_paired(sweep_eta, eta_grid, param_name=r"$\eta$")

#### Neighbor context-to-item-input alignment ($\Delta_{FC}$) asymmetry during updating

In [ ]:
m_all, m_fwd, m_bwd = fc_alignment_asymmetry_means(sweep_eta, eta_grid)
colors, norm_c, cmap_c = make_sweep_colors(eta_grid)

fig, ax = plt.subplots(figsize=(6.5, 4))
plot_fc_alignment_asymmetry_curves(
    ax, eta_grid, m_all, m_fwd, m_bwd, colors,
    title=r"$\Delta_{FC}$ vs $\eta$", xlabel=r"$\eta$")
sm = mpl.cm.ScalarMappable(norm=norm_c, cmap=cmap_c); sm.set_array([])
fig.colorbar(sm, ax=ax, pad=0.02).set_label(r"$\eta$")
fig.tight_layout(); plt.show()

---
## 4. Sweep global encoding drift scale $B_{enc}$

In [ ]:
sweep_B_enc_scale = sweep_one_param(
    param_name="B_encD_scale",
    param_grid=B_encD_scale_grid,
    base_params=BASE_PARAMS,
    n_sims=n_sims,
    collect_diagnostics=True,
    recency_k=3,
)

### Behavioral metrics: accuracy

In [ ]:
plot_recall_accuracy(sweep_B_enc_scale, B_encD_scale_grid, r"$B_{enc}$ scale")

### Behavioral metrics: SPC

In [ ]:
plot_spc_sweep(sweep_B_enc_scale, B_encD_scale_grid, r"$B_{enc}$ scale")

### Behavioral metrics: PFR

In [ ]:
plot_pfr_heatmap(sweep_B_enc_scale, B_encD_scale_grid, r"$B_{enc}$ scale")

### Behavioral metrics: lag-CRP

In [ ]:
plot_lag_crp_sweep(sweep_B_enc_scale, B_encD_scale_grid, r"$B_{enc}$ scale")

### Behavioral metrics: conditional lag rates

In [ ]:
plot_directional_lag_rates(sweep_B_enc_scale, B_encD_scale_grid, r"$B_{enc}$ scale")

### Behavioral metrics: unconditional lag summaries

In [ ]:
plot_unconditional_lag_summaries(sweep_B_enc_scale, B_encD_scale_grid, r"$B_{enc}$ scale")

### Model Internal Diagnostics ($B_{enc}$ scale)

Encoding strength — most informative internals are matrix magnitude
(does scaling actually change stored association strength?), evidence
profiles, neighbor asymmetries, and recall truncation.

#### $M_{FC}$ norms

In [ ]:
plot_matrix_norms_sweep(sweep_B_enc_scale, B_encD_scale_grid, r"$B_{enc}$ scale", matrix_key="net_w_fc")

#### $M_{CF}$ norms

In [ ]:
plot_matrix_norms_sweep(sweep_B_enc_scale, B_encD_scale_grid, r"$B_{enc}$ scale", matrix_key="net_w_cf")

#### $M_{FC}$ lag-strength profile

In [ ]:
plot_w_fc_lag_strength_profile_sweep(sweep_B_enc_scale, B_encD_scale_grid, r"$B_{enc}$ scale", matrix_key="net_w_fc")

#### $M_{FC}$ forward/backward lag asymmetry

In [ ]:
plot_w_fc_forward_backward_lag_asymmetry_sweep(sweep_B_enc_scale, B_encD_scale_grid, r"$B_{enc}$ scale", matrix_key="net_w_fc")

#### Mean $f_{in}(i)$ by serial position

In [ ]:
plot_mean_evidence_by_pos(sweep_B_enc_scale, B_encD_scale_grid, param_name=r"$B_{enc}$ scale")

#### Neighbor item-evidence asymmetry during selection

In [ ]:
plot_item_evidence_asymmetry_paired(sweep_B_enc_scale, B_encD_scale_grid, param_name=r"$B_{enc}$ scale")

#### Neighbor context-to-item-input alignment ($\Delta_{FC}$) asymmetry during updating

In [ ]:
m_all, m_fwd, m_bwd = fc_alignment_asymmetry_means(sweep_B_enc_scale, B_encD_scale_grid)
colors, norm_c, cmap_c = make_sweep_colors(B_encD_scale_grid)

fig, ax = plt.subplots(figsize=(6.5, 4))
plot_fc_alignment_asymmetry_curves(
    ax, B_encD_scale_grid, m_all, m_fwd, m_bwd, colors,
    title=r"$\Delta_{FC}$ vs $B_{encD}$ scale", xlabel=r"$B_{encD}$ scale")
sm = mpl.cm.ScalarMappable(norm=norm_c, cmap=cmap_c); sm.set_array([])
fig.colorbar(sm, ax=ax, pad=0.02).set_label(r"$B_{encD}$ scale")
fig.tight_layout(); plt.show()

#### Cosine similarity $\cos(c, c_{in})$ after update

In [ ]:
plot_cosine_similarity_sweep(sweep_B_enc_scale, B_encD_scale_grid, param_name=r"$B_{enc}$ scale")

#### Mean items recalled per trial

In [ ]:
counts = sweep_recall_counts(sweep_B_enc_scale, B_encD_scale_grid)
param_grid = np.asarray(B_encD_scale_grid, dtype=float)
colors, norm_c, cmap_c = make_sweep_colors(param_grid)

fig, ax = plt.subplots(figsize=(6.5, 4))
line_with_colored_points(ax, param_grid, counts, colors)
ax.set_title(r"Mean items recalled vs $B_{enc}$ scale")
ax.set_xlabel(r"$B_{enc}$ scale"); ax.set_ylabel("mean recalls / trial")
ax.grid(alpha=0.3)
_add_colorbar(fig, ax, norm_c, cmap_c, r"$B_{enc}$ scale")
fig.tight_layout(); plt.show()

---
## To Sweep a New Parameter

```python
# 1. Define grid
my_grid = [0.1, 0.2, 0.3]

# 2. Run sweep
sweep_X = sweep_one_param("my_param", my_grid, BASE_PARAMS, n_sims=n_sims)

# 3. Mandatory behavioral readouts (one per cell)
plot_recall_accuracy(sweep_X, my_grid, r"$X$")
plot_spc_sweep(sweep_X, my_grid, r"$X$")
plot_pfr_heatmap(sweep_X, my_grid, r"$X$")
plot_lag_crp_sweep(sweep_X, my_grid, r"$X$")
plot_directional_lag_rates(sweep_X, my_grid, r"$X$")
plot_unconditional_lag_summaries(sweep_X, my_grid, r"$X$")

# 4. Parameter-specific diagnostics (choose relevant ones)
plot_item_evidence_asymmetry_paired(sweep_X, my_grid, r"$X$")
plot_fc_alignment_asymmetry_sweep(sweep_X, my_grid, r"$X$")
plot_mean_evidence_by_pos(sweep_X, my_grid, r"$X$")
plot_w_fc_lag_strength_profile_sweep(sweep_X, my_grid, r"$X$")
plot_w_fc_forward_backward_lag_asymmetry_sweep(sweep_X, my_grid, r"$X$")
```